# Save-As-You-Go: Streaming DAQ Data to HDF5

This notebook demonstrates versionable's incremental HDF5 persistence.
Data is written to disk as it arrives — no need to hold everything in memory.

> **Requirements:** `plotly` and `jupyterlab`

In [ ]:
from __future__ import annotations

from dataclasses import dataclass, field
from datetime import datetime
from pathlib import Path

import numpy as np
import numpy.typing as npt
import plotly.graph_objects as go
from daq_simulator import DaqSimulator
from plotly.subplots import make_subplots

import versionable
import versionable.hdf5
from versionable import Versionable

## 1. Define the Recording type

A normal Versionable dataclass. Inside a session, all ndarray fields are
automatically backed by resizable HDF5 datasets — no annotation needed.

In [ ]:
@dataclass
class Recording(Versionable, version=1, hash="74a182"):
    """A single DAQ recording with time-series data."""

    name: str = ""
    sampleRate_Hz: float = 0.0
    startTime: datetime = field(default_factory=lambda: datetime.now().astimezone())
    time_s: npt.NDArray[np.float64] = field(default_factory=lambda: np.empty(0))
    voltage_V: npt.NDArray[np.float64] = field(default_factory=lambda: np.empty(0))

## 2. Record the first session

`open()` accepts either a **class** (empty proxy) or an **existing instance**.
Here we pass an instance — only set what you care about, defaults handle the rest.

> The `DaqSimulator` is defined in [`daq_simulator.py`](daq_simulator.py) — a simple
> generator that yields chunks of a sinusoidal signal.

In [3]:
OUTPUT_PATH = Path("recording.h5")

daq = DaqSimulator(frequency_Hz=10.0, amplitude_V=1.0, sampleRate_Hz=1000.0, duration_s=1.0)

rec = Recording(name="10 Hz sinusoid", sampleRate_Hz=daq.sampleRate_Hz)

with versionable.hdf5.open(rec, OUTPUT_PATH, mode="overwrite") as rec:
    for tChunk, vChunk in daq.stream():
        rec.time_s.append(tChunk)
        rec.voltage_V.append(vChunk)

print(f"Wrote {OUTPUT_PATH} ({OUTPUT_PATH.stat().st_size / 1024:.1f} KB)")

Wrote recording.h5 (15.2 KB)


## 3. Load and plot the first session

The file is a standard HDF5 — load it with `versionable.load()`, no special API.

In [4]:
first = versionable.load(Recording, OUTPUT_PATH)

print(f"Name:        {first.name}")
print(f"Sample rate: {first.sampleRate_Hz:.0f} Hz")
print(f"Start time:  {first.startTime.isoformat()}")
print(f"Samples:     {len(first.time_s)}")
print(f"Duration:    {first.time_s[-1] - first.time_s[0]:.3f} s")

fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=first.time_s,
        y=first.voltage_V,
        mode="lines",
        name="First session",
        line=dict(color="#636EFA"),
    )
)
fig.update_layout(
    title="After first session (1.0 s)",
    xaxis_title="Time (s)",
    yaxis_title="Voltage (V)",
    template="plotly_white",
    height=350,
)
fig.show()

Name:        10 Hz sinusoid
Sample rate: 1000 Hz
Start time:  2026-04-10T21:02:10.935045+00:00
Samples:     1000
Duration:    0.999 s


## 4. Resume and append more data

Reopen with `mode="resume"` — existing data is restored, and new appends
continue from where we left off.

In [5]:
# Figure out where the first session ended
prev = versionable.load(Recording, OUTPUT_PATH)
startOffset = float(prev.time_s[-1]) + 1.0 / prev.sampleRate_Hz
print(f"Resuming from {len(prev.time_s)} samples (t={startOffset:.4f} s)")

# Append another 0.5 seconds
daq2 = DaqSimulator(frequency_Hz=10.0, amplitude_V=1.0, sampleRate_Hz=1000.0, duration_s=0.5)

with versionable.hdf5.open(Recording, OUTPUT_PATH, mode="resume") as rec:
    for tChunk, vChunk in daq2.stream(startOffset_s=startOffset):
        rec.time_s.append(tChunk)
        rec.voltage_V.append(vChunk)

    print(f"Total samples: {len(rec.time_s)}")

Resuming from 1000 samples (t=1.0000 s)
Total samples: 1500


## 5. Load and plot the combined result

The file now contains both sessions. The boundary is seamless.

In [6]:
combined = versionable.load(Recording, OUTPUT_PATH)
boundary = len(first.time_s)

print(f"Total samples: {len(combined.time_s)}")
print(f"Total duration: {combined.time_s[-1] - combined.time_s[0]:.3f} s")

fig = make_subplots(
    rows=2,
    cols=1,
    subplot_titles=("After first session (1.0 s)", "After resume (+0.5 s)"),
    vertical_spacing=0.15,
)

# Top: first session only
fig.add_trace(
    go.Scatter(
        x=first.time_s,
        y=first.voltage_V,
        mode="lines",
        name="First session",
        line=dict(color="#636EFA"),
    ),
    row=1,
    col=1,
)

# Bottom: both sessions, colored differently
fig.add_trace(
    go.Scatter(
        x=combined.time_s[:boundary],
        y=combined.voltage_V[:boundary],
        mode="lines",
        name="First session",
        line=dict(color="#636EFA"),
        showlegend=False,
    ),
    row=2,
    col=1,
)
fig.add_trace(
    go.Scatter(
        x=combined.time_s[boundary:],
        y=combined.voltage_V[boundary:],
        mode="lines",
        name="Resumed session",
        line=dict(color="#EF553B"),
    ),
    row=2,
    col=1,
)

fig.update_xaxes(title_text="Time (s)", row=2, col=1)
fig.update_yaxes(title_text="V", row=1, col=1)
fig.update_yaxes(title_text="V", row=2, col=1)
fig.update_layout(template="plotly_white", height=550)
fig.show()

Total samples: 1500
Total duration: 1.499 s


## Cleanup

In [ ]:
OUTPUT_PATH.unlink(missing_ok=True)  # Delete the file when done
print(f"Deleted {OUTPUT_PATH.name}")